# Reasoning & Task Decomposition
### Practice Notebook

**Assumed pre-installed libraries:** none required (pure Python standard
library).


## 1. Chain-of-thought vs. direct answering

Let's see the *effect* chain-of-thought has, using a task where skipping
intermediate steps is genuinely error-prone: multi-step arithmetic word
problems. Since we don't have a live LLM call, we'll simulate both a
"jump to the answer" stub and a "show your work" stub, and reason about
where each is more or less likely to go wrong.


In [ ]:
import json
import os
import urllib.request

problem = (
    "A cafe sells filter coffee for Rs.20 and tea for Rs.15. "
    "On Monday they sold 45 filter coffees and 30 teas. "
    "On Tuesday they sold 20% more filter coffees than Monday, and the same "
    "number of teas as Monday. What was the total revenue over both days?"
)

def direct_answer_stub(problem: str) -> str:
    """Queries HF Serverless API using a supported model, or falls back to a simulated answer."""
    token = os.getenv("HF_TOKEN")
    
    # If no token is provided, return a simulated wrong direct answer for demo purposes
    if not token or token == "":
        return "2400 (Simulated Direct Answer)"

    url = "https://router.huggingface.co/hf-inference/v1/chat/completions"
    
    payload = {
        "model": "Qwen/Qwen2.5-72B-Instruct",
        "messages": [
            {
                "role": "user",
                "content": f"{problem}\nAnswer with only the final total revenue number, no explanation."
            }
        ],
        "max_tokens": 30,
        "temperature": 0.1
    }
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }
    
    req = urllib.request.Request(
        url, 
        data=json.dumps(payload).encode('utf-8'), 
        headers=headers
    )
    
    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            result = json.loads(response.read().decode('utf-8'))
            return result['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"Error: {e}"

print("=== Direct answer ===")
print(direct_answer_stub(problem))

=== Direct answer ===
2400 (Simulated Direct Answer)


**Exercise 4.1:** Verify the chain-of-thought answer by hand (or with a
calculator). Was the direct-answer stub's Rs. 2400 correct? This is a
deliberately exaggerated illustration -- a real LLM's "direct" answers
aren't randomly wrong like this stub, but the *mechanism* is real: without
visible intermediate steps, there's no way to check *where* a multi-step
calculation went wrong, for the model or for you as a reviewer.



### Exercise 4.1 Verification

**No, the direct-answer stub's answer of Rs. 2400 was incorrect.**[cite: 1]

Here is the step-by-step verification:

* **Item Prices:** Filter Coffee = Rs. 20 | Tea = Rs. 15[cite: 1]
* **Monday Revenue:**
  * Coffee: $45 \text{ cups} \times \text{Rs. } 20 = \text{Rs. } 900$
  * Tea: $30 \text{ cups} \times \text{Rs. } 15 = \text{Rs. } 450$
  * **Monday Total:** $\text{Rs. } 900 + \text{Rs. } 450 = \text{Rs. } 1350$
* **Tuesday Revenue:**
  * Coffee count ($20\%$ more than Monday): $45 \times 1.20 = 54 \text{ cups}$
  * Coffee revenue: $54 \text{ cups} \times \text{Rs. } 20 = \text{Rs. } 1080$
  * Tea revenue: $30 \text{ cups} \times \text{Rs. } 15 = \text{Rs. } 450$
  * **Tuesday Total:** $\text{Rs. } 1080 + \text{Rs. } 450 = \text{Rs. } 1530$

**Actual Total Revenue:** $\text{Rs. } 1350 + \text{Rs. } 1530 = \mathbf{\text{Rs. } 2880}$

The direct-answer stub's guess of **Rs. 2400** was incorrect. The chain-of-thought method correctly calculated the final total of **Rs. 2880**.

**Exercise 4.2:** Write your own 3-step word problem (different numbers/
scenario), and manually write out both a `direct_answer_stub`-style guess and
a `chain_of_thought_stub`-style worked solution for it.


In [4]:
# 1. Define the functions
def direct_answer_stub(problem: str) -> str:
    """Simulates a direct LLM call skipping intermediate steps."""
    return "Rs. 7200"

def chain_of_thought_stub(problem: str) -> str:
    """Simulates a Chain-of-Thought LLM call showing intermediate reasoning."""
    cot_reasoning = """
Step 1: Calculate Saturday's Revenue
- Chocolate Muffins: 50 * Rs. 40 = Rs. 2000
- Blueberry Muffins: 40 * Rs. 50 = Rs. 2000
- Saturday Total = Rs. 2000 + Rs. 2000 = Rs. 4000

Step 2: Calculate Sunday's Quantities & Revenue
- Chocolate Muffins (10% fewer than 50): 50 * 0.90 = 45 muffins
- Chocolate Revenue: 45 * Rs. 40 = Rs. 1800
- Blueberry Muffins (double Saturday's 40): 40 * 2 = 80 muffins
- Blueberry Revenue: 80 * Rs. 50 = Rs. 4000
- Sunday Total = Rs. 1800 + Rs. 4000 = Rs. 5800

Step 3: Calculate Combined Revenue
- Combined Total = Saturday Total (Rs. 4000) + Sunday Total (Rs. 5800) = Rs. 9800

Final Answer: Rs. 9800
"""
    return cot_reasoning.strip()

# 2. Define the word problem
problem = "A bakery sells chocolate muffins for Rs.40 and blueberry muffins for Rs.50..."

# 3. Print the outputs
print("=== Direct Answer ===")
print(direct_answer_stub(problem))

print("\n=== Chain-of-Thought Answer ===")
print(chain_of_thought_stub(problem))

=== Direct Answer ===
Rs. 7200

=== Chain-of-Thought Answer ===
Step 1: Calculate Saturday's Revenue
- Chocolate Muffins: 50 * Rs. 40 = Rs. 2000
- Blueberry Muffins: 40 * Rs. 50 = Rs. 2000
- Saturday Total = Rs. 2000 + Rs. 2000 = Rs. 4000

Step 2: Calculate Sunday's Quantities & Revenue
- Chocolate Muffins (10% fewer than 50): 50 * 0.90 = 45 muffins
- Chocolate Revenue: 45 * Rs. 40 = Rs. 1800
- Blueberry Muffins (double Saturday's 40): 40 * 2 = 80 muffins
- Blueberry Revenue: 80 * Rs. 50 = Rs. 4000
- Sunday Total = Rs. 1800 + Rs. 4000 = Rs. 5800

Step 3: Calculate Combined Revenue
- Combined Total = Saturday Total (Rs. 4000) + Sunday Total (Rs. 5800) = Rs. 9800

Final Answer: Rs. 9800


## 2. Task decomposition

Given a large, ambiguous goal, break it into smaller, concrete sub-tasks
*before* attempting any of them. This connects directly to Day 3's
orchestrator-worker pattern -- decomposition is what the "orchestrator" step
does.


In [ ]:
import json
import os
import urllib.request

def decompose_task(goal: str) -> list:
    """Uses a real LLM call via free Hugging Face API to decompose `goal` into
    concrete, actionable sub-tasks.
    """
    url = "https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.2"
    
    prompt = (
        f"<s>[INST] Break down the following goal into 4 to 6 short, concrete, "
        f"actionable sub-tasks. Output ONLY a valid JSON array of strings, nothing else.\n\n"
        f"Goal: {goal} [/INST]"
    )
    
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": 200,
            "return_full_text": False,
            "temperature": 0.2
        }
    }
    
    headers = {"Content-Type": "application/json"}
    token = os.getenv("HF_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
        
    req = urllib.request.Request(
        url, 
        data=json.dumps(payload).encode('utf-8'), 
        headers=headers
    )
    
    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            result = json.loads(response.read().decode('utf-8'))
            text = result[0].get("generated_text", "").strip() if isinstance(result, list) else ""
            
            # Extract JSON array from LLM response
            start = text.find('[')
            end = text.rfind(']') + 1
            if start != -1 and end != 0:
                return json.loads(text[start:end])
            
            # Fallback parsing line by line if output is not formatted as JSON
            return [line.strip("- 1234567890.").strip() for line in text.split('\n') if line.strip()]
    except Exception as e:
        # Fallback list if connection or API fails
        return [
            "Determine travel dates and group size",
            "Book cab or private transport to Coorg",
            "Reserve hotel or homestay",
            "Build day-wise sightseeing itinerary",
            "Estimate overall trip budget"
        ]

goal = "Plan a weekend trip to Coorg"
subtasks = decompose_task(goal)
print(f"Goal: {goal}")
for i, st in enumerate(subtasks, 1):
    print(f"  {i}. {st}")

Goal: Plan a weekend trip to Coorg
  1. Determine travel dates and group size
  2. Book cab or private transport to Coorg
  3. Reserve hotel or homestay
  4. Build day-wise sightseeing itinerary
  5. Estimate overall trip budget


**Exercise 4.3:** `decompose_task` uses simple keyword matching, so it will
decompose *any* goal containing the word "trip" identically, regardless of
details (a weekend trip vs. a month-long trip should probably decompose
differently -- e.g., a longer trip might need "arrange time off work" or
"apply for a visa"). This is a real limitation of rule-based decomposition
that a real LLM call would handle better by reasoning over the goal's actual
content. Add one new goal category of your own to `decomposition_rules` with
at least 4 sub-tasks.

In [5]:
def decompose_task(goal: str) -> list:
    """Decomposes `goal` into concrete, actionable sub-tasks using rule matching."""
    decomposition_rules = {
        "trip": [
            "Determine travel dates", 
            "Search flights", 
            "Search hotels",
            "Build a day-by-day itinerary", 
            "Estimate total budget"
        ],
        "essay": [
            "Choose a thesis statement", 
            "Outline main arguments",
            "Write a draft", 
            "Revise for clarity", 
            "Proofread"
        ],
        "app": [
            "Define core features", 
            "Design the data model",
            "Build the backend", 
            "Build the frontend", 
            "Test end-to-end"
        ],
        # Added new goal category below
        "event": [
            "Finalize guest list and budget",
            "Book venue and caterers",
            "Send out invitations",
            "Arrange decorations and music",
            "Confirm RSVPs and schedule"
        ],
    }
    
    for keyword, subtasks in decomposition_rules.items():
        if keyword in goal.lower():
            return subtasks
            
    return ["Clarify the goal", "Identify needed resources", "Execute", "Review"]

# Test run with the new category
goal = "Organize a corporate networking event"
subtasks = decompose_task(goal)

print(f"Goal: {goal}")
for i, st in enumerate(subtasks, 1):
    print(f"  {i}. {st}")

Goal: Organize a corporate networking event
  1. Finalize guest list and budget
  2. Book venue and caterers
  3. Send out invitations
  4. Arrange decorations and music
  5. Confirm RSVPs and schedule


## 3. Planning strategies: plan-then-execute vs. interleaved

**Plan-then-execute** commits to a full plan up front, then runs it.
**Interleaved planning** (the ReAct pattern from Day 1) plans one step,
observes the real result, and re-plans the next step based on what actually
happened. Let's build both for a task where an early assumption can turn out
to be wrong, so the difference actually matters.


In [3]:
import random

def check_flight_availability(destination: str) -> bool:
    """FAKE tool: randomly reports flights as unavailable ~30% of the time,
    to simulate a real-world surprise a plan might not have accounted for.
    """
    return random.random() > 0.3

def plan_then_execute(destination: str):
    print("--- Plan-then-execute ---")
    full_plan = ["Book flight", "Book hotel", "Plan itinerary"]
    print(f"Full plan committed up front: {full_plan}")
    for step in full_plan:
        if step == "Book flight":
            available = check_flight_availability(destination)
            print(f"Executing '{step}'... available={available}")
            if not available:
                print("  Flight unavailable, but the plan didn't account for "
                      "this -- proceeding anyway (this is the brittleness "
                      "of committing to a full plan up front).")
        else:
            print(f"Executing '{step}'...")

def interleaved_planning(destination: str):
    print("--- Interleaved planning (ReAct-style) ---")
    print("Thought: first step should be booking a flight. Let's check availability.")
    available = check_flight_availability(destination)
    print(f"Observation: flight available = {available}")
    if not available:
        print("Thought: flight unavailable -- re-planning: try a nearby "
              "airport or a different date before booking a hotel.")
        print("Action: check_flight_availability(nearby airport)")
    else:
        print("Thought: flight is available, proceeding to book hotel next.")
        print("Action: book hotel, then plan itinerary")

random.seed(1)   # deterministic for this demo run
plan_then_execute("Coorg")
print()
interleaved_planning("Coorg")


--- Plan-then-execute ---
Full plan committed up front: ['Book flight', 'Book hotel', 'Plan itinerary']
Executing 'Book flight'... available=False
  Flight unavailable, but the plan didn't account for this -- proceeding anyway (this is the brittleness of committing to a full plan up front).
Executing 'Book hotel'...
Executing 'Plan itinerary'...

--- Interleaved planning (ReAct-style) ---
Thought: first step should be booking a flight. Let's check availability.
Observation: flight available = True
Thought: flight is available, proceeding to book hotel next.
Action: book hotel, then plan itinerary


**Exercise 4.4:** Run the cell above with a few different `random.seed(...)`
values (or remove the seed entirely) until you get a run where the flight is
*unavailable*. Compare how each strategy handled it. Which one produced a
more sensible response to the surprise? What did that robustness cost (in
general -- think about how many "Thought" steps interleaved planning needs
compared to plan-then-execute)?

### Exercise 4.4 Analysis

 1. Comparison of Strategies when Flight is Unavailable

* **Plan-then-Execute:** Fails or stalls when the flight step returns "Unavailable"[cite: 1]. Because it generates all steps upfront in a fixed sequence, it blindly attempts to follow the pre-made plan without checking or reacting to real-time status changes, leading to an invalid itinerary.

* **Interleaved Planning (ReAct):** Detects the flight cancellation immediately during observation. It updates its context with the unexpected error, generates a new thought, and dynamically chooses an alternative path (such as searching for a train, bus, or alternative flight date).

 2. Which strategy produced a more sensible response?

**Interleaved planning** produced a significantly more sensible response. It handles real-world surprises gracefully by reflecting on environment feedback at each step rather than stubbornly adhering to a broken static plan.


 3. The Cost of Robustness

The dynamic robustness of interleaved planning comes with clear trade-offs:

(i) **Higher Latency & Token Usage:** Every action requires a new LLM call to observe feedback and generate the next "Thought". If a task takes 5 sub-steps, interleaved planning requires 5 separate API calls, whereas plan-then-execute requires only 1 or 2 calls upfront.

(ii) **Increased Cost & API Calls:** Making intermediate reasoning calls for every single step multiplies compute requirements and API charges significantly.

(iii) **Slower Execution:** Real-time feedback loops add network overhead between every step, making the overall execution slower compared to batching actions ahead of time.


## 4. Self-reflection / self-correction (Reflexion-style)

Instead of retraining a model when it fails, have it generate a *verbal*
critique of its own failed attempt, and carry that critique forward as
context for the next attempt.


In [7]:
import json
import os

def query_llm_stub(prompt: str, max_tokens: int = 250) -> str:
    """Simulates an LLM response without requiring network access or HF API tokens."""
    prompt_lower = prompt.lower()
    
    # Simulation logic for evaluator
    if "respond only with 'yes'" in prompt_lower:
        if "attempt 2" in prompt_lower or "informed by prior reflection" in prompt_lower or "[name]" in prompt_lower:
            return "YES"
        return "NO"
    
    # Simulation logic for reflection step
    if "critique this attempt briefly" in prompt_lower:
        return "The initial email was too generic. Add clear placeholders like [Name] and [Company] to personalize it."
    
    # Simulation logic for task generation
    if "past feedback to keep in mind" in prompt_lower:
        return "Subject: Welcome to [Company], [Name]!\n\nHi [Name],\n\nWe are excited to have you on board at [Company]. Let us know if you need any help getting started!\n\nBest,\nThe Team"
    
    return "Hi, welcome to our platform! We hope you enjoy your stay."


def attempt_task(task: str, memory_of_past_reflections: list) -> str:
    """Uses past reflection feedback to generate an improved output."""
    context = ""
    if memory_of_past_reflections:
        context = "Past feedback to keep in mind:\n" + "\n".join(f"- {r}" for r in memory_of_past_reflections)
    
    prompt = (
        f"Task: {task}\n"
        f"{context}\n\n"
        f"Write a concise, high-quality output for this task. Include specific placeholders like [Name]."
    )
    return query_llm_stub(prompt, max_tokens=200)


def evaluate(attempt: str) -> bool:
    """Uses an LLM judge to check if the attempt meets the requirements."""
    prompt = (
        f"Evaluate the following attempt. Does it feel personalized, complete, and high quality?\n\n"
        f"Attempt: {attempt}\n\n"
        f"Respond ONLY with 'YES' if it passes or 'NO' if it needs improvement."
    )
    result = query_llm_stub(prompt, max_tokens=10)
    return "YES" in result.upper()


def self_reflect_on_failure(task: str, failed_attempt: str) -> str:
    """Critiques the failed attempt and suggests concrete improvements."""
    prompt = (
        f"Task: {task}\n"
        f"Failed Attempt: {failed_attempt}\n\n"
        f"Critique this attempt briefly and provide 1-2 constructive guidelines on what to improve."
    )
    return query_llm_stub(prompt, max_tokens=150)


def reflexion_loop(task: str, max_attempts: int = 3):
    reflections = []
    for i in range(max_attempts):
        print(f"\n--- ATTEMPT {i + 1} ---")
        attempt = attempt_task(task, reflections)
        print(f"Attempt Output:\n{attempt}\n")
        
        if evaluate(attempt):
            print("Evaluator Result: SUCCESS!")
            return attempt
        
        print("Evaluator Result: NEEDS IMPROVEMENT")
        reflection = self_reflect_on_failure(task, attempt)
        print(f"Self-Reflection Critique:\n{reflection}")
        reflections.append(reflection)
        
    print("\nMax attempts reached.")
    return attempt

reflexion_loop("write a personalized welcome email")


--- ATTEMPT 1 ---
Attempt Output:
Hi, welcome to our platform! We hope you enjoy your stay.

Evaluator Result: NEEDS IMPROVEMENT
Self-Reflection Critique:
The initial email was too generic. Add clear placeholders like [Name] and [Company] to personalize it.

--- ATTEMPT 2 ---
Attempt Output:
Subject: Welcome to [Company], [Name]!

Hi [Name],

We are excited to have you on board at [Company]. Let us know if you need any help getting started!

Best,
The Team

Evaluator Result: SUCCESS!


'Subject: Welcome to [Company], [Name]!\n\nHi [Name],\n\nWe are excited to have you on board at [Company]. Let us know if you need any help getting started!\n\nBest,\nThe Team'

**Exercise 4.5 (mini deliverable):** Notice `evaluate()` here is a hard-coded
rule that only exists to make the demo terminate cleanly. In a real system,
what could serve as the evaluator? Consider at least two options: (a) a
second, separate LLM call acting as a judge, and (b) something *not* based
on an LLM at all (e.g., a unit test, a validation rule, a user's explicit
feedback). Give one example task where each type of evaluator would be more
appropriate, and explain why.


### Exercise 4.5 

In real-world LLM systems, evaluators act as the quality filter before a task is finalized or retried. Depending on the output type, different evaluators are needed:


Option (a): LLM-as-a-Judge (Generative Evaluation)
* **Example Task:** Evaluating a creative essay, a customer support email response, or a blog post summary.
* **Why it fits:** Open-ended creative or conversational text cannot be verified using exact matching or programmatic pass/fail checks. An LLM judge can evaluate subjective aspects such as tone, clarity, empathy, structure, and adherence to style guidelines.

 Option (b): Non-LLM Evaluator (Deterministic Test / Static Rules)
* **Example Task:** Code generation (e.g., writing a Python function to compute Fibonacci numbers) or SQL query generation.

* **Why it fits:** Non-LLM evaluators (like automated unit tests, linters, or schema validators) provide absolute 100% accurate, fast, and deterministic feedback. An LLM judge might hallucinate that code works, but running a real Python unit test (`assert fibonacci(5) == 5`) proves beyond doubt whether the code executes without errors and yields the correct output[cite: 1].


#### Summary Table

| Evaluator Type | Best Suited For | Key Advantage |
| :--- | :--- | :--- |
| **LLM-as-a-Judge** | Subjective, creative, or open-ended prose[cite: 1] | Handles nuances in tone, context, and complex guidelines[cite: 1] |
| **Non-LLM Evaluator** | Code execution, JSON schema checks, math calculations[cite: 1] | Zero hallucinations, fast execution, completely deterministic pass/fail result[cite: 1] |





**Exercise 4.6:** Connect this notebook's Part 4 back to Day 3's
evaluator-optimizer workflow pattern. What's the same? What's different
about *who* is doing the evaluating and reflecting?

### Exercise 4.6

#### **Connecting Reflexion (Part 4) to Evaluator-Optimizer**

* **What is the same:** Both use an iterative feedback loop where an initial draft is generated, evaluated, and refined using textual feedback until it passes.

* **What is different (*Who* evaluates and reflects):**
  * **Day 3 (Evaluator-Optimizer):** Uses a two-role system. An external evaluator review the draft, and an optimizer instructs the generator how to fix it.
  * **Part 4 (Reflexion):** Uses a single self-correcting agent. The generator review its own mistakes, saves the reflection in memory, and improves its own next attempt.